In [1]:
import sys
import plotly.graph_objects as go
from pathlib import Path
import pandas as pd
import numpy as np
import re
# --- Data Loading ---

files = [
    'prices_round_5_day_2.csv',
    'prices_round_5_day_3.csv',
    'prices_round_5_day_4.csv'
]

prices = []
for file in files:
    try:
        df = pd.read_csv(file, sep=';')
        day_match = re.search(r'day_(-?\d+)', file)
        if day_match:
            df['day'] = int(day_match.group(1))
        prices.append(df)
    except FileNotFoundError:
        print(f"Warning: {file} not found.")

df_total = pd.concat(prices, ignore_index=True)
df_total = df_total.sort_values(by=['day', 'timestamp']).reset_index(drop=True)

In [ ]:
# ── Comprehensive Statistical Matrix Suite ───────────────────────────────────

# Dictionary to store all matrix types
market_stats = {
    "returns_corr": {},
    "returns_cov":  {},
    "price_corr":   {},
    "autocorr_lag": {}
}

# 1. Category-Specific Matrices
for category, products in CATEGORIES.items():
    # Pivot for price and returns
    cat_df = df_total[df_total['product'].isin(products)].pivot_table(
        index=['day', 'timestamp'], columns='product', values='mid_price'
    ).sort_index()
    
    cat_returns = cat_df.pct_change().dropna()

    if cat_returns.empty:
        continue

    # Standard Return Matrices (Best for Alpha)
    market_stats["returns_corr"][category] = cat_returns.corr()
    market_stats["returns_cov"][category]  = cat_returns.cov()
    
    # Price Correlation (Best for identifying hard-pegs or co-integration)
    market_stats["price_corr"][category] = cat_df.corr()

    # Autocorrelation Matrix (Diagonal = self-lag, Off-diagonal = Lead/Lag)
    # This shows if Product A's move now predicts Product B's move in 1 tick
    lagged_returns = cat_returns.shift(1)
    cross_autocorr = cat_returns.corrwith(lagged_returns) 
    market_stats["autocorr_lag"][category] = cross_autocorr

# 2. Master Cross-Category Correlation Matrix
# This helps identify if 'Robots' are actually correlated with 'Microchips'
master_pivot = df_total.pivot_table(
    index=['day', 'timestamp'], columns='product', values='mid_price'
).pct_change().dropna()

master_corr = master_pivot.corr()

# ── Visualization: Master Heatmap ─────────────────────────────────────────────
fig_master = px.imshow(
    master_corr,
    title="Master Market Correlation Matrix (All Products)",
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    height=1000, width=1000
)
fig_master.update_layout(margin=dict(l=150, r=10, b=150, t=50))
fig_master.show()

# ── Summary Analysis Function ────────────────────────────────────────────────
def get_top_relationships(matrix, n=5):
    """Flattens a matrix to find the strongest non-diagonal relationships."""
    upper_tri = matrix.where(np.triu(np.ones(matrix.shape), k=1).astype(bool))
    flat = upper_tri.unstack().dropna().sort_values(ascending=False)
    return flat

print("\n" + "="*60)
print(" TOP 5 HIGHLY CORRELATED PAIRS (MARKET-WIDE)")
print("="*60)
print(get_top_relationships(master_corr).head(5))

print("\n" + "="*60)
print(" TOP 5 MEAN-REVERTING CANDIDATES (Strongest Negative Autocorr)")
print("="*60)
# Flattening the autocorrelation series
all_autocorr = pd.concat(market_stats["autocorr_lag"].values()).sort_values()
print(all_autocorr.head(5))


 ORACLE PnL RANKING | limit=10 | Dynamic Spread: (Ask1-Bid1)/2
Rank  Product                                    Category            Total PnL
-------------------------------------------------------------------------------------
1     PEBBLES_XL                                 Pebbles            2,186,494.0
2     MICROCHIP_SQUARE                           Microchips         1,343,767.0
3     MICROCHIP_TRIANGLE                         Microchips         1,049,270.0
4     PEBBLES_XS                                 Pebbles            1,006,773.0
5     PEBBLES_S                                  Pebbles             962,995.0
6     PEBBLES_M                                  Pebbles             960,774.0
7     PEBBLES_L                                  Pebbles             953,579.0
8     MICROCHIP_OVAL                             Microchips          948,802.0
9     MICROCHIP_RECTANGLE                        Microchips          921,151.0
10    SLEEP_POD_POLYESTER                        Sleep P

In [11]:
# ── Enhanced Category Engine: Covariance & Lead-Lag Analysis ────────────────

for category, products in CATEGORIES.items():
    # 1. Data Preparation
    cat_df = df_total[df_total['product'].isin(products)].pivot_table(
        index=['day', 'timestamp'], 
        columns='product', 
        values='mid_price'
    ).sort_index()

    cat_returns = cat_df.pct_change().dropna()
    if cat_returns.empty:
        continue

    print(f"\n{'#'*85}")
    print(f" CATEGORY ANALYSIS: {category.upper()}")
    print(f"{'#'*85}")

    # 2. Matrix Calculations
    corr_matrix = cat_returns.corr()
    cov_matrix = cat_returns.cov() * 1e6 # Scaled for visualization clarity
    
    # Lead-Lag Matrix calculation: Corr(Asset_i at t, Asset_j at t-1)
    # Interpretation: If [A, B] is high, B leads A.
    lead_lag_matrix = pd.DataFrame(index=cat_returns.columns, columns=cat_returns.columns)
    for col_leader in cat_returns.columns:
        for row_follower in cat_returns.columns:
            # Correlation between follower return and leader's previous return
            lead_lag_matrix.loc[row_follower, col_leader] = \
                cat_returns[row_follower].corr(cat_returns[col_leader].shift(1))

    # 3. Visualization Suite
    if len(products) > 1:
        # --- Correlation Heatmap ---
        fig_corr = px.imshow(
            corr_matrix, text_auto=".2f", color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
            title=f"{category} - Correlation Matrix (Concurrent)", 
            labels=dict(color="Corr"), width=500, height=450
        )
        fig_corr.show()

        # --- Covariance Heatmap (NEW) ---
        fig_cov = px.imshow(
            cov_matrix, text_auto=".2f", color_continuous_scale='Viridis',
            title=f"{category} - Covariance Matrix (x10^-6)", 
            labels=dict(color="Cov"), width=500, height=450
        )
        fig_cov.show()

        # --- Lead-Lag Heatmap (NEW) ---
        # This addresses the "PEBBLES_XL is lagging" hypothesis
        fig_lag = px.imshow(
            lead_lag_matrix.astype(float), text_auto=".2f", color_continuous_scale='Picnic',
            title=f"{category} - Lead-Lag Matrix (Row follows Column at t-1)",
            labels=dict(x="Leader (t-1)", y="Follower (t)", color="Lag-Corr"),
            width=500, height=450
        )
        fig_lag.show()

    # 4. Ticker Tear Sheet (kept for reference)
    print(f"\n{'Ticker':<30} | {'Vol (Annal)':>10} | {'AutoCorr':>10} | {'Skew':>8} | {'Kurt':>8}")
    print("-" * 85)
    for product in products:
        if product in cat_returns.columns:
            s = cat_returns[product]
            print(f"{product:<30} | {s.std()*1000:>10.2f} | {s.autocorr():>10.3f} | {s.skew():>8.2f} | {s.kurt():>8.2f}")


#####################################################################################
 CATEGORY ANALYSIS: GALAXY SOUNDS
#####################################################################################



Ticker                         | Vol (Annal) |   AutoCorr |     Skew |     Kurt
-------------------------------------------------------------------------------------
GALAXY_SOUNDS_DARK_MATTER      |       1.00 |     -0.012 |    -0.02 |    -0.01
GALAXY_SOUNDS_BLACK_HOLES      |       1.00 |     -0.017 |     0.01 |     0.04
GALAXY_SOUNDS_PLANETARY_RINGS  |       1.01 |     -0.003 |     0.02 |     0.01
GALAXY_SOUNDS_SOLAR_WINDS      |       1.01 |     -0.007 |     0.01 |     0.02
GALAXY_SOUNDS_SOLAR_FLAMES     |       1.00 |     -0.012 |     0.01 |    -0.01

#####################################################################################
 CATEGORY ANALYSIS: SLEEP PODS
#####################################################################################



Ticker                         | Vol (Annal) |   AutoCorr |     Skew |     Kurt
-------------------------------------------------------------------------------------
SLEEP_POD_SUEDE                |       1.00 |     -0.007 |    -0.02 |    -0.01
SLEEP_POD_LAMB_WOOL            |       1.00 |      0.004 |     0.00 |     0.03
SLEEP_POD_POLYESTER            |       1.00 |     -0.001 |     0.02 |    -0.03
SLEEP_POD_NYLON                |       1.00 |      0.001 |    -0.00 |     0.02
SLEEP_POD_COTTON               |       1.01 |     -0.002 |    -0.02 |    -0.01

#####################################################################################
 CATEGORY ANALYSIS: MICROCHIPS
#####################################################################################



Ticker                         | Vol (Annal) |   AutoCorr |     Skew |     Kurt
-------------------------------------------------------------------------------------
MICROCHIP_CIRCLE               |       1.00 |     -0.005 |    -0.02 |     0.02
MICROCHIP_OVAL                 |       1.50 |     -0.007 |    -0.01 |     0.07
MICROCHIP_SQUARE               |       1.51 |     -0.022 |     0.01 |     0.04
MICROCHIP_RECTANGLE            |       1.50 |     -0.003 |     0.01 |    -0.00
MICROCHIP_TRIANGLE             |       1.49 |     -0.008 |     0.01 |     0.01

#####################################################################################
 CATEGORY ANALYSIS: PEBBLES
#####################################################################################



Ticker                         | Vol (Annal) |   AutoCorr |     Skew |     Kurt
-------------------------------------------------------------------------------------
PEBBLES_XS                     |       2.14 |     -0.019 |    -0.00 |     0.32
PEBBLES_S                      |       1.71 |      0.009 |     0.01 |     0.10
PEBBLES_M                      |       1.48 |     -0.004 |     0.02 |     0.05
PEBBLES_L                      |       1.49 |      0.007 |    -0.00 |     0.05
PEBBLES_XL                     |       2.36 |      0.008 |     0.01 |     0.24

#####################################################################################
 CATEGORY ANALYSIS: ROBOTS
#####################################################################################


...


Ticker                         | Vol (Annal) |   AutoCorr |     Skew |     Kurt
-------------------------------------------------------------------------------------
UV_VISOR_YELLOW                |       1.00 |      0.003 |     0.00 |     0.01
UV_VISOR_AMBER                 |       1.01 |     -0.003 |    -0.01 |    -0.02
UV_VISOR_ORANGE                |       1.00 |      0.001 |    -0.03 |     0.01
UV_VISOR_RED                   |       1.00 |     -0.004 |    -0.00 |    -0.03
UV_VISOR_MAGENTA               |       1.01 |     -0.003 |     0.01 |     0.05

#####################################################################################
 CATEGORY ANALYSIS: TRANSLATORS
#####################################################################################



Ticker                         | Vol (Annal) |   AutoCorr |     Skew |     Kurt
-------------------------------------------------------------------------------------
TRANSLATOR_SPACE_GRAY          |       1.00 |      0.008 |     0.02 |    -0.02
TRANSLATOR_ASTRO_BLACK         |       1.00 |     -0.007 |    -0.01 |     0.03
TRANSLATOR_ECLIPSE_CHARCOAL    |       1.00 |     -0.007 |    -0.02 |     0.02
TRANSLATOR_GRAPHITE_MIST       |       1.00 |     -0.004 |     0.03 |    -0.06
TRANSLATOR_VOID_BLUE           |       1.00 |     -0.009 |    -0.01 |     0.01

#####################################################################################
 CATEGORY ANALYSIS: PANELS
#####################################################################################



Ticker                         | Vol (Annal) |   AutoCorr |     Skew |     Kurt
-------------------------------------------------------------------------------------
PANEL_1X2                      |       1.01 |     -0.002 |    -0.02 |    -0.01
PANEL_2X2                      |       1.00 |     -0.010 |     0.01 |    -0.02
PANEL_1X4                      |       1.00 |     -0.001 |    -0.00 |     0.04
PANEL_2X4                      |       1.00 |     -0.002 |    -0.00 |    -0.05
PANEL_4X4                      |       1.01 |     -0.006 |     0.00 |    -0.01

#####################################################################################
 CATEGORY ANALYSIS: OXYGEN SHAKES
#####################################################################################



Ticker                         | Vol (Annal) |   AutoCorr |     Skew |     Kurt
-------------------------------------------------------------------------------------
OXYGEN_SHAKE_MORNING_BREATH    |       1.01 |     -0.005 |     0.02 |     0.01
OXYGEN_SHAKE_EVENING_BREATH    |       1.17 |     -0.118 |     0.04 |    10.50
OXYGEN_SHAKE_MINT              |       1.00 |     -0.003 |     0.01 |    -0.05
OXYGEN_SHAKE_CHOCOLATE         |       1.12 |     -0.082 |    -0.07 |    10.76
OXYGEN_SHAKE_GARLIC            |       1.01 |     -0.003 |     0.01 |    -0.01

#####################################################################################
 CATEGORY ANALYSIS: SNACK PACKS
#####################################################################################



Ticker                         | Vol (Annal) |   AutoCorr |     Skew |     Kurt
-------------------------------------------------------------------------------------
SNACKPACK_CHOCOLATE            |       0.67 |     -0.031 |     0.01 |     0.07
SNACKPACK_VANILLA              |       0.65 |     -0.027 |     0.00 |     0.06
SNACKPACK_PISTACHIO            |       0.55 |     -0.025 |    -0.00 |    -0.03
SNACKPACK_STRAWBERRY           |       0.76 |     -0.014 |    -0.00 |    -0.02
SNACKPACK_RASPBERRY            |       0.80 |     -0.017 |     0.00 |    -0.01


In [12]:
# ── AI-REVEAL: Exporting Global Statistical State ─────────────────────────────
import json

ai_data_package = {}

for category, products in CATEGORIES.items():
    cat_df = df_total[df_total['product'].isin(products)].pivot_table(
        index=['day', 'timestamp'], columns='product', values='mid_price'
    ).sort_index()
    cat_returns = cat_df.pct_change().dropna()

    if cat_returns.empty: continue

    # Calculate matrices
    corr = cat_returns.corr().to_dict()
    # Scale cov for precision in export
    cov = (cat_returns.cov() * 1e6).to_dict()
    
    # Lead-Lag (t-1)
    ll_dict = {}
    for col_leader in cat_returns.columns:
        ll_dict[col_leader] = {
            row_follower: float(cat_returns[row_follower].corr(cat_returns[col_leader].shift(1)))
            for row_follower in cat_returns.columns
        }

    # Ticker Stats
    ticker_stats = {}
    for p in products:
        if p in cat_returns.columns:
            s = cat_returns[p]
            ticker_stats[p] = {
                "vol_annal": float(s.std() * 1000),
                "autocorr": float(s.autocorr(lag=1)),
                "skew": float(s.skew()),
                "kurt": float(s.kurtosis())
            }

    ai_data_package[category] = {
        "correlations": corr,
        "covariances_scaled": cov,
        "lead_lag_signals": ll_dict,
        "ticker_metrics": ticker_stats
    }

# ── PRINT FOR AI UPLOAD ──────────────────────────────────────────────────────
print("--- START AI DATA PACKAGE ---")
# Using indent=2 makes it readable for both humans and AIs
print(json.dumps(ai_data_package, indent=2))
print("--- END AI DATA PACKAGE ---")

--- START AI DATA PACKAGE ---
{
  "Galaxy Sounds": {
    "correlations": {
      "GALAXY_SOUNDS_BLACK_HOLES": {
        "GALAXY_SOUNDS_BLACK_HOLES": 1.0,
        "GALAXY_SOUNDS_DARK_MATTER": 0.00011605300153369568,
        "GALAXY_SOUNDS_PLANETARY_RINGS": 0.001215211301241774,
        "GALAXY_SOUNDS_SOLAR_FLAMES": 0.0019372093477228662,
        "GALAXY_SOUNDS_SOLAR_WINDS": 0.00894170037673907
      },
      "GALAXY_SOUNDS_DARK_MATTER": {
        "GALAXY_SOUNDS_BLACK_HOLES": 0.00011605300153369568,
        "GALAXY_SOUNDS_DARK_MATTER": 1.0,
        "GALAXY_SOUNDS_PLANETARY_RINGS": 0.007440499427896093,
        "GALAXY_SOUNDS_SOLAR_FLAMES": 0.004495662770864101,
        "GALAXY_SOUNDS_SOLAR_WINDS": -0.001317938771841247
      },
      "GALAXY_SOUNDS_PLANETARY_RINGS": {
        "GALAXY_SOUNDS_BLACK_HOLES": 0.001215211301241774,
        "GALAXY_SOUNDS_DARK_MATTER": 0.007440499427896093,
        "GALAXY_SOUNDS_PLANETARY_RINGS": 1.0,
        "GALAXY_SOUNDS_SOLAR_FLAMES": 0.01297412560992671,
